In [1]:
import warnings
warnings.filterwarnings("ignore")
import regional_mom6 as rmom6

import os
from pathlib import Path
from dask.distributed import Client
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cftime

client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 1,Total memory: 7.81 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42695,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45867,Total threads: 1
Dashboard: http://127.0.0.1:33761/status,Memory: 7.81 GiB
Nanny: tcp://127.0.0.1:45249,


Specifying the home directory:

In [2]:
path = "/scratch/gpfs/CDEUTSCH/wchu/"

## Grid setup

### Topography file `topog.nc`

Initially, working with the seamount configuration, we crop the dataset to match the geometry (40 x 2): 
```
ds.h.isel(ny=slice(5,7),nx=slice(140,180))
```
Now that we are trying a larger domain, use
```
ds.h.isel(ny=slice(5,55),nx=slice(140,240))
```
to match the aspect ratio of the final run.

In [8]:
ds = xr.open_dataset(path+"/MOM6-examples-topo/ocean_only/seamount/sigma/NEPTUNE/neptune_grid.nc")
ds = ds.rename_dims({'xi_rho':'nx','eta_rho':'ny'}) # Name dimensions same as in topog.nc file
topog = ds.h.isel(nx=slice(1,-1),ny=slice(1,-1)).rename('depth')  # Crop edges to match MOM6 grid
topog.plot(x='nx',y='ny')
topog.to_netcdf(path+"/MOM6-examples-topo/ocean_only/seamount/sigma_low/INPUT/topog.nc")

### Vertical resolution `thickness.nc`

In [11]:
ds = xr.open_dataset(path+"/MOM6-examples-topo/ocean_only/seamount/sigma/NEPTUNE/neptune_init.nc") # Initialization in NEPTUNE
topog = xr.open_dataset(path+"/MOM6-examples-topo/ocean_only/seamount/sigma_low/INPUT/topog.nc")       # Topography file, generated above

eta = (topog.depth*ds.Cs_w[::-1])   # Cs_w is normalized interface height from -1 to 0, take absolute value
eta = eta.rename({'s_w':'nz'})        # Rename the coordinates
eta = eta.to_dataset(name='eta')      # Name the variable, MOM6 is expecting 'eta'
eta = eta.transpose('nz','ny','nx')   # Switch order of dimensions since MOM6 is expecting [nz,ny,nx]
eta.to_netcdf(path+"/MOM6-examples-topo/ocean_only/seamount/sigma_low/INPUT/thickness.nc")

### Setting layer coordinates `layer_coord.nc`

In [14]:
ds = xr.open_dataset(path+"/MOM6-examples-topo/ocean_only/seamount/sigma/NEPTUNE/neptune_init.nc") # initialization in NEPTUNE
np.abs(ds.Cs_w[::-1]).diff(dim='s_w')

# layer = (np.abs(ds.Cs_r))                                   # Cs_r is normalized layer height from -1 to 0, take absolute value
# layer = layer[::-1].rename({'s_rho':'zl'}).rename('Layer')  # Rename vertical coordinate and variable name
# interface = (np.abs(ds.Cs_w))
# interface = interface[::-1].rename({'s_w':'zi'}).rename('Interface')
# layer_coord = xr.merge([layer,interface])
# layer_coord.to_netcdf(path+"/MOM6-examples-topo/ocean_only/seamount/sigma/INPUT/layer_coord.nc")

<xarray.DataArray 'Cs_w' (s_w: 80)> Size: 640B
array([0.00018696, 0.00056141, 0.00093742, 0.00131602, 0.00169826,
       0.00208515, 0.00247773, 0.00287698, 0.00328391, 0.00369949,
       0.00412467, 0.00456038, 0.00500751, 0.00546691, 0.00593939,
       0.00642572, 0.00692658, 0.00744258, 0.00797426, 0.00852206,
       0.0090863 , 0.00966715, 0.01026468, 0.01087874, 0.01150905,
       0.01215506, 0.01281604, 0.01349097, 0.01417856, 0.0148772 ,
       0.01558496, 0.01629953, 0.01701822, 0.01773792, 0.0184551 ,
       0.01916577, 0.01986546, 0.02054926, 0.02121175, 0.02184707,
       0.02244893, 0.02301062, 0.02352506, 0.02398491, 0.02438262,
       0.02471052, 0.02496098, 0.02512656, 0.02520014, 0.02517515,
       0.02504576, 0.02480709, 0.02445546, 0.02398861, 0.02340592,
       0.02270865, 0.02190004, 0.02098553, 0.0199728 , 0.01887176,
       0.01769452, 0.01645523, 0.01516977, 0.01385545, 0.01253055,
       0.01121377, 0.00992365, 0.00867796, 0.00749303, 0.00638318,
       0.00536018, 0.00443286, 0.00360682, 0.00288433, 0.00226446,
       0.00174329, 0.00131435, 0.00096918, 0.00069797, 0.00049017])
Dimensions without coordinates: s_w

## Initial conditions

### Temperature and salinity `initial_TS.nc`

In [12]:
ds = xr.open_dataset(path+"/MOM6-examples-topo/ocean_only/seamount/sigma/NEPTUNE/neptune_init.nc")

TS = ds[['temp','salt']]
TS = TS.rename({'s_rho':'nz','eta_rho':'ny','xi_rho':'nx'})           # Rename the coordinates
TS = TS.isel(nx=slice(1,-1),ny=slice(1,-1),nz=slice(None, None, -1))  # Crop edges, reverse order of vertical
TS.to_netcdf(path+"/MOM6-examples-topo/ocean_only/seamount/sigma_low/INPUT/initial_TS.nc")

### Random wind surface forcing `surface_frc.nc`

In [10]:
surface_frc = xr.open_dataset(path+"/MOM6-examples-topo/ocean_only/seamount/sigma/NEPTUNE/neptune_frc.nc")
surface_frc = surface_frc[["sustr","svstr","swrad"]]
surface_frc = surface_frc.rename({'xi_rho':'xh','eta_rho':'yh','xi_u':'xq','eta_v':'yq'})

surface_frc['frc_time'] = xr.date_range(start=cftime.DatetimeJulian(1,1,1,1,0,0,0), periods=surface_frc.frc_time.size, freq="D", use_cftime=True, calendar="cftime.DatetimeJulian")
surface_frc = surface_frc.rename({'frc_time':'time'})
surface_frc = surface_frc.isel(xh=slice(1,-1),yh=slice(1,-1),xq=slice(0,-1),yq=slice(0,-1))
surface_frc.to_netcdf(path+"/MOM6-examples-topo/ocean_only/seamount/sigma_low/INPUT/surface_frc.nc",unlimited_dims='time',encoding={"time": {"dtype": "double"}})  